In [ ]:
#############################################################
#
#############################################################

In [1]:
import pickle
import numpy as np
import os
import math
import numpy as np
import pygad
import random
from tqdm import tqdm

def count_certain_elements(all_lists, t = 1):
    certain_elements = 0
    for a in range(len(all_lists[0])):
        t_help = 0
        b = 0
        while t_help < t and b < len(all_lists):
            if all_lists[b][a]:
                t_help += 1
            b = b + 1
        if t_help >= t:
            certain_elements += 1
    #print("cc=" + str(count_certain_elements))
    return (certain_elements / len(all_lists[0]))

def load_eval(path_to_dir, add_names_to_files = ""):
    files = os.listdir(path_to_dir)
    numpy_embedding_test = []
    for f_id in range(len(files)):
        with open(path_to_dir + "/" + files[f_id], "rb") as f:
            numpy_embedding_test.append(pickle.load(f))
        files[f_id] = add_names_to_files + files[f_id]
    return (numpy_embedding_test, files)

In [2]:
#############################################################
# Load data
#############################################################
(numpy_embedding_test, numpy_embedding_test_files) = load_eval("numpy_embedding_test")
(numpy_embedding_train, numpy_embedding_train_files) = load_eval("numpy_embedding_train")
(numpy_embedding_test_base, numpy_embedding_test_base_files) = load_eval("numpy_embedding_test_base", "base_")
(numpy_embedding_train_base, numpy_embedding_train_base_files) = load_eval("numpy_embedding_train_base", "base_")
numpy_test = []
numpy_train = []
numpy_test.extend(numpy_embedding_test)
numpy_test.extend(numpy_embedding_test_base)
numpy_train.extend(numpy_embedding_train)
numpy_train.extend(numpy_embedding_train_base)

'model_sbert_TH_78000k_.pkl'

In [3]:
# --- GLOBAL PROGRESS BAR ---
pbar = None

n = 20   # size of population
m = len(numpy_test)  # length of each list
num_generations = 10
eval_repeats = 20
random.seed(123)
np.random.seed(123)

def on_generation(ga_instance):
    global pbar
    #print("Generation = " + str(pbar.n))
    pbar.update(1)


def fitness_func(ga_instance, solution, solution_idx):
    my_solution = []
    # penalty on zero solution
    if np.sum(solution) < 0.5:
        return -1000
    for a in range(len(solution)):
        if solution[a] == 1:
            my_solution.append(numpy_train[a])
    tt = int(len(my_solution) / 2) + 1
    fitness = count_certain_elements(my_solution, tt)
    
    # the higher fitness (sum of items) the better
    return fitness

def eval_res(sol):
    my_solution = []
    for a in range(len(sol)):
        if sol[a] == 1:
            my_solution.append(numpy_test[a])
    tt = int(len(my_solution) / 2) + 1
    fitness = count_certain_elements(my_solution, tt)
    
    # the higher fitness (sum of items) the better
    return fitness


In [4]:
def fitness_func(solution):
    my_solution = []
    # penalty on zero solution
    if np.sum(solution) < 0.5:
        return -1000
    for a in range(len(solution)):
        if solution[a] == 1:
            my_solution.append(numpy_train[a])
    tt = int(len(my_solution) / 2) + 1
    fitness = count_certain_elements(my_solution, tt)
    
    # the higher fitness (sum of items) the better
    return fitness

In [5]:
import numpy as np
def simulated_annealing(func,
                        n,
                        T0=10.0,
                        Tmin=1e-4,
                        alpha=0.995,
                        steps_per_temp=100):
    # losowe rozwiązanie początkowe
    x = np.random.randint(0, 2, size=n, dtype=np.uint8)
    fx = func(x)
    best_x = x.copy()
    best_fx = fx
    T = T0
    while T > Tmin:
        print(T)
        for _ in range(steps_per_temp):
            # generacja sąsiada
            y = x.copy()
            idx = np.random.randint(n)
            y[idx] = 1 - y[idx]
            fy = func(y)
            delta = fy - fx
            # akceptacja
            if delta >= 0:
                x = y
                fx = fy
            else:
                p = np.exp(delta / T)
                if np.random.rand() < p:
                    x = y
                    fx = fy
            # aktualizacja najlepszego rozwiązania
            if fx > best_fx:
                best_fx = fx
                best_x = x.copy()
        T *= alpha
    return best_x, best_fx


In [6]:
for foo in range(eval_repeats):
    best_x, best_f = simulated_annealing(
        fitness_func,
        n=10,
        alpha=0.75,
        Tmin=0.1,
        steps_per_temp=10
    )
    
    print(best_x)
    print(best_f)
    my_str = str(best_x) + ";" + str(np.where(best_x == 1)[0]) + ";" + str(best_f) + ";" + str(eval_res(best_x)) + "\n"
    with open("./ga_res/simulated_annealing.txt", "a") as f:
        f.writelines(my_str)

10.0
7.5
5.625
4.21875
3.1640625
2.373046875
1.77978515625
1.3348388671875
1.001129150390625
0.7508468627929688
0.5631351470947266
0.4223513603210449
0.3167635202407837
0.23757264018058777
0.17817948013544083
0.13363461010158062
0.10022595757618546
[1 0 0 1 1 0 0 0 0 0]
0.9685101098524102
10.0
7.5
5.625
4.21875
3.1640625
2.373046875
1.77978515625
1.3348388671875
1.001129150390625
0.7508468627929688
0.5631351470947266
0.4223513603210449
0.3167635202407837
0.23757264018058777
0.17817948013544083
0.13363461010158062
0.10022595757618546
[1 0 0 1 1 0 0 0 0 0]
0.9685101098524102
10.0
7.5
5.625
4.21875
3.1640625
2.373046875
1.77978515625
1.3348388671875
1.001129150390625
0.7508468627929688
0.5631351470947266
0.4223513603210449
0.3167635202407837
0.23757264018058777
0.17817948013544083
0.13363461010158062
0.10022595757618546
[1 0 1 1 1 1 0 0 0 0]
0.9696456341853782
10.0
7.5
5.625
4.21875
3.1640625
2.373046875
1.77978515625
1.3348388671875
1.001129150390625
0.7508468627929688
0.5631351470947266

In [14]:

#[1 0 0 0 1 1 0 0 0 0];[0 4 5];0.9720130819207798;0.9933162901912902

[1 0 1 1 1 1 0 0];[0 2 3 4 5];0.9696456341853782;0.9938186813186813



In [7]:
import numpy as np
from tqdm import tqdm
def binary_pso(
    func,
    n,
    swarm_size=50,
    max_iter=300,
    w=0.7,
    c1=1.5,
    c2=1.5,
):
    """
    Binary Particle Swarm Optimization.

    Parameters
    ----------
    func : callable
        Fitness function to maximize.
    n : int
        Number of binary variables.
    swarm_size : int
        Number of particles.
    max_iter : int
        Number of iterations.

    Returns
    -------
    best_x : ndarray
        Best binary solution.
    best_f : float
        Best fitness value.
    """
    # losowe pozycje
    X = np.random.randint(
        0, 2,
        size=(swarm_size, n),
        dtype=np.uint8
    )
    # losowe prędkości
    V = np.random.uniform(
        -1.0,
        1.0,
        size=(swarm_size, n)
    )
    # personal best
    P = X.copy()
    P_fit = np.array([
        func(x) for x in X
    ])
    # global best
    g_idx = np.argmax(P_fit)
    G = P[g_idx].copy()
    G_fit = P_fit[g_idx]
    # główna pętla
    for _ in tqdm(range(max_iter)):

        r1 = np.random.rand(
            swarm_size, n
        )

        r2 = np.random.rand(
            swarm_size, n
        )

        # aktualizacja prędkości
        V = (
            w * V
            + c1 * r1 * (P - X)
            + c2 * r2 * (G - X)
        )

        # sigmoid
        prob = 1.0 / (
            1.0 + np.exp(-V)
        )

        # aktualizacja pozycji
        X = (
            np.random.rand(
                swarm_size, n
            ) < prob
        ).astype(np.uint8)

        # fitness
        fitness = np.array([
            func(x) for x in X
        ])

        # aktualizacja personal best
        improved = fitness > P_fit

        P[improved] = X[improved]
        P_fit[improved] = fitness[improved]

        # aktualizacja global best
        idx = np.argmax(P_fit)

        if P_fit[idx] > G_fit:
            G_fit = P_fit[idx]
            G = P[idx].copy()

    return G, G_fit

In [8]:
for foo in range(eval_repeats):
    best_x, best_f = binary_pso(
             fitness_func,
             n=10,
             swarm_size=20,
             max_iter=20
         )
    print(best_x)
    print(best_f)
    my_str = str(best_x) + ";" + str(np.where(best_x == 1)[0]) + ";" + str(best_f) + ";" + str(eval_res(best_x)) + "\n"
    print(my_str)
    with open("./ga_res/binary_pos.txt", "a") as f:
        f.writelines(my_str)
#[1 0 0 0 1 1 0 0 0 0];[0 4 5];0.9720130819207798;0.9933162901912902

100%|██████████| 20/20 [05:04<00:00, 15.21s/it]


[1 0 1 1 0 1 0 0 1 0]
0.957455404798122
[1 0 1 1 0 1 0 0 1 0];[0 2 3 5 8];0.957455404798122;0.9921525234025234



100%|██████████| 20/20 [05:12<00:00, 15.62s/it]


[1 0 0 0 0 1 0 1 0 0]
0.9581401217180016
[1 0 0 0 0 1 0 1 0 0];[0 5 7];0.9581401217180016;0.9906199124949125



100%|██████████| 20/20 [05:04<00:00, 15.20s/it]


...

[1 0 0 0 1 1 0 1 1 0]
0.9516849281472524
[1 0 0 0 1 1 0 1 1 0];[0 4 5 7 8];0.9516849281472524;0.9886039886039886



100%|██████████| 20/20 [05:07<00:00, 15.35s/it]


[1 0 1 0 1 1 0 1 0 0]
0.9644748164519188
[1 0 1 0 1 1 0 1 0 0];[0 2 4 5 7];0.9644748164519188;0.9926358363858364



100%|██████████| 20/20 [04:56<00:00, 14.83s/it]

[1 0 1 1 1 1 0 1 1 0]
0.9606585474077864
[1 0 1 1 1 1 0 1 1 0];[0 2 3 4 5 7 8];0.9606585474077864;0.9926739926739927



In [34]:
my_str = ""
df = pd.read_csv("./ga_res/simulated_annealing.txt", sep=";", header=None)
my_mean = np.round(np.mean(np.array(df.iloc[:,2])),3)
my_std = np.round(np.std(np.array(df.iloc[:,2])),3)
my_str += str(my_mean) + "$\\pm$" + str(my_std)

tt_ga = np.mean(np.array(df.iloc[:,3]))
my_mean = np.round(np.mean(np.array(df.iloc[:,3])),3)
my_std = np.round(np.std(np.array(df.iloc[:,3])),3)

my_str += " & " + str(my_mean) + "$\\pm$" + str(my_std)
print(my_str)
my_str = ""
df = pd.read_csv("./ga_res/binary_pos.txt", sep=";", header=None)
my_mean = np.round(np.mean(np.array(df.iloc[:,2])),3)
my_std = np.round(np.std(np.array(df.iloc[:,2])),3)
my_str += str(my_mean) + "$\\pm$" + str(my_std)

tt_ga = np.mean(np.array(df.iloc[:,3]))
my_mean = np.round(np.mean(np.array(df.iloc[:,3])),3)
my_std = np.round(np.std(np.array(df.iloc[:,3])),3)

my_str += " & " + str(my_mean) + "$\\pm$" + str(my_std)
print(my_str)

0.969$\pm$0.004 & 0.993$\pm$0.001
0.959$\pm$0.007 & 0.991$\pm$0.002
